# Inspect base trajectory

Plots a `base_trajectory.csv` (`t, x, y, yaw, err_x, err_y, err_yaw[, stage]`) against the Task 2 approach waypoints.

Left: full path. Right: 10 cm box around the final goal. Red = waypoint pose (point + heading). Blue = logged base path (points connected in time, every `PLOT_EVERY_N`th sample). Yellow = final robot position.

Below: `err_x`, `err_y`, `err_yaw` vs time from super-fine start through the end of the log.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

ROOT = Path("/home/ubuntu/workspace/camelo-ebim")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from camelo.control.approach import (
    STAGE_FINE,
    TASK2_APPROACH_WAYPOINTS,
    PulseSettleTrim,
)

TRAJ_CSV: Path = Path(
    "/home/ubuntu/workspace/camelo-ebim/outputs/eval/20260813_152507/episode_000/base_trajectory.csv"
)

PLOT_EVERY_N = 50
ZOOM_BOX_M = 0.10
OVERVIEW_TICK_M = 0.20
ZOOM_TICK_M = 0.005


In [ ]:
def every_nth(df: pd.DataFrame, n: int) -> pd.DataFrame:
    """Keep every n-th row, always including the last sample."""
    if df.empty:
        return df
    n = max(1, int(n))
    idx = list(range(0, len(df), n))
    last_i = len(df) - 1
    if idx[-1] != last_i:
        idx.append(last_i)
    return df.iloc[idx]


def draw_heading(ax, x, y, yaw, *, color: str, tick_m: float, lw: float = 1.4):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    yaw = np.asarray(yaw, dtype=float)
    dx = tick_m * np.cos(yaw)
    dy = tick_m * np.sin(yaw)
    ax.plot(
        np.vstack([x, x + dx]),
        np.vstack([y, y + dy]),
        color=color,
        lw=lw,
        solid_capstyle="round",
        zorder=4,
    )


csv_path = Path(TRAJ_CSV)
df = pd.read_csv(csv_path)
need = {"t", "x", "y", "yaw", "err_x", "err_y", "err_yaw"}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"{csv_path} missing columns {sorted(missing)}")

wps = TASK2_APPROACH_WAYPOINTS
goal = wps[-1]
wp_x = np.array([wp.x for wp in wps])
wp_y = np.array([wp.y for wp in wps])
wp_yaw = np.array([wp.yaw for wp in wps])
shown = every_nth(df, PLOT_EVERY_N)
body = shown.iloc[:-1]
final = df.iloc[-1]
zoom_half = ZOOM_BOX_M / 2.0

print(
    f"{csv_path}\n"
    f"{len(df)} samples, plotted {len(shown)} (every {PLOT_EVERY_N}, +last)  "
    f"t={df['t'].iloc[0]:.2f}→{df['t'].iloc[-1]:.2f} s  "
    f"final err=({final['err_x']*1000:+.1f} mm, {final['err_y']*1000:+.1f} mm, "
    f"{final['err_yaw']:+.4f} rad)"
)


In [ ]:
fig, (ax, axz) = plt.subplots(
    1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [1.35, 1]}
)

ax.plot(
    shown["x"], shown["y"],
    color="tab:blue", lw=1.4, alpha=0.9, zorder=2,
    marker="o", ms=4, label="trajectory",
)
ax.scatter(
    [final["x"]], [final["y"]],
    c="yellow", s=80, zorder=7, edgecolors="0.15", linewidths=0.8,
    label="final pose",
)

ax.plot(wp_x, wp_y, color="tab:red", ls="--", lw=1.0, alpha=0.7, zorder=1)
ax.scatter(wp_x, wp_y, c="tab:red", s=40, zorder=5, label="waypoints")
draw_heading(ax, wp_x, wp_y, wp_yaw, color="tab:red", tick_m=OVERVIEW_TICK_M, lw=2.0)
for wp in wps:
    ax.annotate(
        wp.name, (wp.x, wp.y), textcoords="offset points", xytext=(6, 6),
        color="tab:red", fontsize=9,
    )

zoom_box = Rectangle(
    (goal.x - zoom_half, goal.y - zoom_half),
    ZOOM_BOX_M,
    ZOOM_BOX_M,
    fill=False,
    ec="0.3",
    lw=1.0,
    ls=":",
    zorder=6,
)
ax.add_patch(zoom_box)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("approach trajectory")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

axz.plot(
    shown["x"], shown["y"],
    color="tab:blue", lw=1.4, alpha=0.9, zorder=2,
    marker="o", ms=5,
)
axz.scatter(
    [final["x"]], [final["y"]],
    c="yellow", s=90, zorder=7, edgecolors="0.15", linewidths=0.8,
    label="final pose",
)
axz.scatter([goal.x], [goal.y], c="tab:red", s=50, zorder=5)
draw_heading(
    axz, [goal.x], [goal.y], [goal.yaw],
    color="tab:red", tick_m=ZOOM_TICK_M, lw=2.2,
)
axz.annotate(
    goal.name, (goal.x, goal.y), textcoords="offset points", xytext=(6, 6),
    color="tab:red", fontsize=9,
)
axz.set_xlim(goal.x - zoom_half, goal.x + zoom_half)
axz.set_ylim(goal.y - zoom_half, goal.y + zoom_half)
axz.set_aspect("equal", adjustable="box")
axz.set_xlabel("x (m)")
axz.set_ylabel("y (m)")
axz.set_title(f"{ZOOM_BOX_M*100:.0f} cm box around {goal.name}")
axz.legend(loc="best")
axz.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


In [ ]:
def finegrained_span(frame: pd.DataFrame) -> tuple[float, float]:
    """Return (t_start, t_done) of super-fine. t_done is handover, not CSV EOF.

    With a ``stage`` column: last ``finegrained_start_position`` sample.
    Without: the 50 Hz approach loop logs ~4–5 rows per sim tick, the 20 Hz
    policy loop ~2 — the drop is when manipulation starts.
    """
    if "stage" in frame.columns and (frame["stage"] == STAGE_FINE).any():
        fine_t = frame.loc[frame["stage"] == STAGE_FINE, "t"]
        return float(fine_t.iloc[0]), float(fine_t.iloc[-1])

    xy = np.hypot(frame["err_x"], frame["err_y"])
    near = xy <= 0.03
    if not near.any():
        raise ValueError("no near-goal samples to locate super-fine")
    t_start = float(frame.loc[near.to_numpy().argmax(), "t"])
    copies = frame.groupby("t").size().sort_index()
    after = copies[copies.index >= t_start]
    # 50 Hz approach logs ~4–5 rows per sim tick; 20 Hz policy ~2.
    # Bin so a single sparse tick during approach cannot look like handover.
    binned = after.groupby(lambda t: np.floor(t * 4.0) / 4.0).median()
    policy_bins = binned[binned < 3.0]
    if policy_bins.empty:
        print("no 50→20 Hz drop — using last near-goal sample as t_done")
        t_done = float(frame.loc[near, "t"].iloc[-1])
    else:
        t_done = float(policy_bins.index[0])
        print(
            "no stage column — t_done = 50→20 Hz handover after |xy_err|≤3 cm "
            f"(t={t_done:.2f} s); re-run eval for exact super-fine"
        )
    return t_start, t_done


t0, t_done = finegrained_span(df)
# From super-fine start through end of log (incl. rollout after handover).
window = df[df["t"] >= t0].copy()
if window.empty:
    raise ValueError("no super-fine samples in this CSV")
window = window.groupby("t", sort=True).last().reset_index()

t_rel = window["t"] - t0
t_done_rel = float(t_done - t0)
at_done = window.iloc[(window["t"] - t_done).abs().argmin()]
tol_xy, tol_yaw = PulseSettleTrim(goal.x, goal.y, goal.yaw).final_tols()
specs = (
    ("err_x", "err_x (m)", tol_xy),
    ("err_y", "err_y (m)", tol_xy),
    ("err_yaw", "err_yaw (rad)", tol_yaw),
)

fig_e, axes_e = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
for i, (ax_e, (col, ylabel, gate)) in enumerate(zip(axes_e, specs)):
    y = window[col].to_numpy(dtype=float)
    ax_e.plot(t_rel, y, color="tab:blue", lw=1.0)
    ax_e.axhline(0.0, color="0.4", lw=0.8)
    ax_e.axhspan(-gate, gate, color="tab:red", alpha=0.08, zorder=0)
    ax_e.axhline(gate, color="tab:red", ls="--", lw=0.9, label=f"±{gate:g} tol")
    ax_e.axhline(-gate, color="tab:red", ls="--", lw=0.9)
    ax_e.axvline(
        t_done_rel,
        color="tab:green",
        lw=1.5,
        label="finegrained done" if i == 0 else None,
    )
    # Half-range is at least 2× the tol (error bound); grow if |err| exceeds that
    # so a hard ±2×tol clip cannot hide the fine-trim transient.
    data_half = float(np.nanmax(np.abs(y))) if y.size else 0.0
    if not np.isfinite(data_half):
        data_half = 0.0
    half = max(2.0 * gate, data_half)
    if half <= 0.0:
        half = 2.0 * gate
    # ax_e.set_ylim(-half, half)

    ax_e.set_ylim(-3.0 * gate, 3.0 * gate)
    ax_e.set_ylabel(ylabel)
    ax_e.grid(True, alpha=0.3)
    ax_e.legend(loc="upper right")

axes_e[-1].set_xlabel("t (s, from super-fine start)")
axes_e[0].set_title(
    f"errors from super-fine start (green = finegrained done at t={t_done:.2f} s)"
)
axes_e[-1].set_xlim(0.0, float(t_rel.iloc[-1]))
fig_e.tight_layout()
plt.show()
print(
    f"super-fine t={t0:.2f}→{t_done:.2f} s, plot through t={window['t'].iloc[-1]:.2f} s  "
    f"err at done=({at_done['err_x']*1000:+.1f} mm, "
    f"{at_done['err_y']*1000:+.1f} mm, {at_done['err_yaw']:+.4f} rad)  "
    f"tol=(±{tol_xy:g} m, ±{tol_yaw:g} rad)"
)


In [ ]:
# Per-episode base path length during teleop demos:
# scripts/debugging/inspect_demonstration_rollouts.ipynb